# PHerc0846A Inference

Forward-only inference with a flat depth-13 surface. `DEPTH_OFFSET = -1` selects slices 9-16; change the offset immediately below `CHECKPOINT` in the settings cell. Saved checkpoint geometry and normalization are replayed, while this offset explicitly overrides saved offsets.

Run the setup cells, then any individual segment cell. The final cell processes the remaining PHerc0846A quick patches into one `quick_patches` subfolder, excluding segments with individual cells. Figures contain prediction and the composite of the selected input slices. Volumes and masks must already be assembled; this notebook does not reverse volumes, detect surfaces, or modify labels.

In [ ]:
CHECKPOINT = "/vesuvius/models/finetune_letters/refine_baseline_m1/final.pth"
DEPTH_OFFSET = -1

FLAT_DEPTH = 13
EVAL_STRIDE = 32
PREVIEW_DOWNSCALE = 8
DEVICE = "auto"

import os
import sys
from pathlib import Path

REPO = Path("/vesuvius")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)


In [ ]:
import gc
import json
import tempfile
import types

import cv2
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

from utils.config import Config, ScrollConfig, resolve_run_config
from utils.fast_figure import colorize, gray_rgb, save_panel_grid
from utils.model import create_model
from utils.norm import (
    ANCHOR_GAP_LEVEL, ANCHOR_PAPYRUS_LEVEL, UNIFIED_CACHE_PATH,
    compute_norm, compute_surface_anchors, load_cached_norm,
)
from utils.platform import get_zarr_dir
from utils.visualizer import TensorboardVisualizer, predict_tiles

Image.MAX_IMAGE_PIXELS = None
MODEL_PATH = Path(CHECKPOINT).resolve()
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint is missing: {MODEL_PATH}")
RUN_CONFIG, RUN_CONFIG_PATH = resolve_run_config(MODEL_PATH)
C = Config()
for section in ("data", "model"):
    for name, value in RUN_CONFIG[section].items():
        if section == "data" and name == "scrolls":
            C.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(getattr(C, section), name):
            setattr(getattr(C, section), name, value)
C.model.compile_model = False
C.tra.supcon = False
C.tra.dann = False
C.device = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
C.data.eval_stride = EVAL_STRIDE
C.data.surface_window_offset = int(DEPTH_OFFSET)
C.data.surface_window_offset_by_scroll = {}
ZARR_DIR = Path(get_zarr_dir())
C.data.zarr_path = str(ZARR_DIR)
DEPTH = int(C.data.depth)
if DEPTH != 8 or not C.model.surface_teacher_input or not C.data.surface_relative_depth_window:
    raise ValueError("Select a literal-surface, eight-slice checkpoint")
if EVAL_STRIDE <= 0 or EVAL_STRIDE > C.model.multitile_grid * C.model.multitile_subtile:
    raise ValueError("Inference stride must be positive and no larger than the prediction center")
WINDOW_START = FLAT_DEPTH - (DEPTH - 1) // 2 + int(DEPTH_OFFSET)
WINDOW_END = WINDOW_START + DEPTH
NORM_MODE = str(C.data.norm_mode)
MODEL_NAME = f"{MODEL_PATH.parent.parent.name}_{MODEL_PATH.parent.name}_{MODEL_PATH.stem}"
OUT_DIR = REPO / "output" / "0846A" / MODEL_NAME / f"offset_{DEPTH_OFFSET:+d}"
CACHE_ROOT = REPO / "_ves_tmp" / "0846A_inference"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_PATCHES = {
    "auto_grown_20260927134928530_abf": {"zid": "20260927134928", "norm_id": 20260927134931},
    "auto_grown_20261012065803387_abf": {"zid": "20261012065803", "norm_id": 20261012065806},
    "auto_grown_20261012073244544_abf": {"zid": "20261012073244", "norm_id": 20261012073246},
    "auto_grown_20261012081237255_abf": {"zid": "20261012081237", "norm_id": 20261012081238},
}
for patch in MAIN_PATCHES.values():
    patch["mask_dir"] = REPO / "masks"
QUICK_ROOT = REPO / "tifxyz" / "quick_patches_from_D" / "PHerc0846A"
if not QUICK_ROOT.is_dir():
    raise FileNotFoundError(QUICK_ROOT)
QUICK_PATCHES = {
    path.name: {"zid": path.name, "mask_dir": REPO / "masks" / "extras"}
    for path in sorted(QUICK_ROOT.iterdir()) if path.is_dir() and (path / "x.tif").is_file()
}
EXTRA_INDIVIDUAL_NAMES = (
    "auto_grown_20260927143929299_abf", "auto_grown_20260927154033403_abf",
)
for name in EXTRA_INDIVIDUAL_NAMES:
    if name not in QUICK_PATCHES:
        raise FileNotFoundError(f"Individual quick-patch mesh is missing: {name}")
INDIVIDUAL_NAMES = (*MAIN_PATCHES, *EXTRA_INDIVIDUAL_NAMES)
PATCHES = {**QUICK_PATCHES, **MAIN_PATCHES}
REMAINING_QUICK_NAMES = tuple(name for name in QUICK_PATCHES if name not in INDIVIDUAL_NAMES)
MODEL = None
print(f"Model: {MODEL_PATH}\nConfig: {RUN_CONFIG_PATH}\nForward slices: {WINDOW_START}-{WINDOW_END - 1}")
print(f"Output: {OUT_DIR}\nIndividual segments: {len(INDIVIDUAL_NAMES)}; remaining quick patches: {len(REMAINING_QUICK_NAMES)}")

In [ ]:
def get_model():
    global MODEL
    if MODEL is None:
        model, parameter_count = create_model(C)
        state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        elif "state_dict" in state:
            state = state["state_dict"]
        state = {name.removeprefix("module.").removeprefix("_orig_mod."): value for name, value in state.items()}
        state = {name: value for name, value in state.items() if not name.startswith(("supcon_head.", "domain_head."))}
        model.load_state_dict(state, strict=True)
        model.eval()
        MODEL = model
        print(f"Loaded {parameter_count:,} parameters on {C.device}")
    return MODEL


def input_paths(name):
    patch = PATCHES[name]
    volume_path = ZARR_DIR / f"{patch['zid']}.zarr"
    mask_path = patch["mask_dir"] / f"{patch['zid']}.png"
    missing = [str(path) for path in (volume_path, mask_path) if not path.exists()]
    if missing:
        raise FileNotFoundError("Assemble these inputs before inference: " + ", ".join(missing))
    return volume_path, mask_path


def flat_norm(patch, volume_path, mask_path, surface_depth, surface_confidence):
    if "norm_id" in patch and FLAT_DEPTH == 13:
        try:
            cached = load_cached_norm(patch["norm_id"], UNIFIED_CACHE_PATH, mode=NORM_MODE)
        except RuntimeError:
            cached = None
        if cached is not None:
            return cached
    if NORM_MODE == "raw255":
        return 0.0, 1.0, 0.0, 255.0
    private_cache = str(CACHE_ROOT / "global_norm.json")
    if NORM_MODE == "global":
        cached = load_cached_norm(patch["zid"], UNIFIED_CACHE_PATH, mode="global")
        cached = cached or load_cached_norm(patch["zid"], private_cache, mode="global")
        return cached or compute_norm(patch["zid"], str(ZARR_DIR), private_cache, mask_dir=str(patch["mask_dir"]))
    if NORM_MODE not in ("surface_anchor", "surface_anchor_knee"):
        raise ValueError(f"Unsupported normalization: {NORM_MODE}")
    attrs_path = volume_path / ".zattrs"
    signature = {
        "volume": str(volume_path.resolve()), "flat_depth": FLAT_DEPTH,
        "attrs_mtime": attrs_path.stat().st_mtime_ns if attrs_path.exists() else volume_path.stat().st_mtime_ns,
        "mask_mtime": mask_path.stat().st_mtime_ns, "shape": list(surface_depth.shape),
    }
    cache_path = CACHE_ROOT / f"{patch['zid']}_flat_anchors.json"
    if cache_path.is_file():
        cached = json.loads(cache_path.read_text())
        if cached.get("signature") == signature:
            return tuple(cached["norm"])
    with tempfile.TemporaryDirectory(dir=CACHE_ROOT) as temporary:
        surface_dir = Path(temporary) / str(patch["zid"])
        surface_dir.mkdir()
        np.save(surface_dir / "depth.npy", surface_depth)
        np.save(surface_dir / "confidence.npy", surface_confidence)
        anchors = compute_surface_anchors(patch["zid"], str(ZARR_DIR), surface_dir=temporary,
                                          mask_dir=str(patch["mask_dir"]))
    span = (anchors["papyrus"] - anchors["gap"]) / (ANCHOR_PAPYRUS_LEVEL - ANCHOR_GAP_LEVEL)
    lower = anchors["gap"] - ANCHOR_GAP_LEVEL * span
    norm = (0.0, 1.0, lower, lower + span)
    cache_path.write_text(json.dumps({"signature": signature, "norm": norm, "anchors": anchors}, indent=2))
    return norm


def load_inputs(name):
    volume_path, mask_path = input_paths(name)
    volume = zarr.open(str(volume_path), mode="r")
    if volume.attrs.get("layers_reversed_from") or volume.attrs.get("facing_reversed"):
        raise ValueError(f"Forward inference requires the original volume: {volume_path}")
    with Image.open(mask_path) as image:
        mask = np.asarray(image.convert("L")) > 0
    height, width = mask.shape
    if tuple(volume.shape[1:]) != (height, width):
        raise ValueError(f"Mask and volume frames differ for {name}: {mask.shape} vs {volume.shape}")
    if not mask.any() or WINDOW_START < 0 or WINDOW_END > volume.shape[0]:
        raise ValueError(f"Invalid mask or depth window for {name}")
    surface_depth = np.where(mask, FLAT_DEPTH, 255).astype(np.uint8)
    surface_confidence = np.where(mask, 255, 0).astype(np.uint8)
    coords = TensorboardVisualizer._gen_tile_coords(
        types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end),
        (0, height), (0, width), mask, z_step=DEPTH,
    )
    if not coords or any(coordinate[0] != 0 for coordinate in coords):
        raise ValueError(f"Expected one nonempty surface-relative pass for {name}")
    norm = flat_norm(PATCHES[name], volume_path, mask_path, surface_depth, surface_confidence)
    return {"volume": volume, "mask": mask, "surface_depth": surface_depth,
            "surface_confidence": surface_confidence, "coords": coords, "norm": norm}

In [ ]:
def render(name, output_subdir=None, preview=True):
    inputs = load_inputs(name)
    model = get_model()
    height, width = inputs["mask"].shape
    prediction = predict_tiles(
        C, model, inputs["volume"], inputs["mask"], inputs["coords"],
        (0, height), (0, width), C.data.d_start, name, *inputs["norm"],
        surface_depth_map=inputs["surface_depth"],
        surface_confidence_map=inputs["surface_confidence"],
        surface_depth_offset=int(DEPTH_OFFSET),
    )
    projection = np.asarray(inputs["volume"][WINDOW_START:WINDOW_END]).max(axis=0)
    composite = np.clip(projection, 0, 255).astype(np.uint8) * inputs["mask"]
    panels = [[
        (f"{name} | forward {WINDOW_START}-{WINDOW_END - 1}",
         colorize(prediction, "gray_r", (0.78, 0.86, 0.95)), cv2.INTER_NEAREST, None),
        (f"composite {WINDOW_START}-{WINDOW_END - 1}", gray_rgb(composite), cv2.INTER_AREA, 9.362),
    ]]
    folder = OUT_DIR if output_subdir is None else OUT_DIR / output_subdir
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / f"{name}.png"
    save_panel_grid(panels, (height, width), str(output_path), gap_px=24)
    if preview:
        with Image.open(output_path) as saved:
            reduced = saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE),
                                    max(1, saved.height // PREVIEW_DOWNSCALE)), Image.Resampling.LANCZOS)
        display(reduced)
    print(f"Saved {output_path}")
    del inputs, prediction, projection, composite, panels
    gc.collect()
    if C.device == "cuda":
        torch.cuda.empty_cache()
    return output_path


def render_remaining_quick_patches():
    for name in REMAINING_QUICK_NAMES:
        input_paths(name)
    failures = {}
    outputs = []
    for index, name in enumerate(REMAINING_QUICK_NAMES, start=1):
        print(f"[{index}/{len(REMAINING_QUICK_NAMES)}] {name}")
        try:
            outputs.append(render(name, output_subdir="quick_patches", preview=False))
        except Exception as error:
            failures[name] = str(error)
            print(f"FAILED {name}: {error}")
    print(f"Saved {len(outputs)} quick-patch figures to {OUT_DIR / 'quick_patches'}")
    if failures:
        raise RuntimeError(f"Quick-patch failures: {failures}")
    return outputs

In [ ]:
render("auto_grown_20260927134928530_abf")

In [ ]:
render("auto_grown_20261012065803387_abf")

In [ ]:
render("auto_grown_20261012073244544_abf")

In [ ]:
render("auto_grown_20261012081237255_abf")

In [ ]:
render("auto_grown_20260927143929299_abf")

In [ ]:
render("auto_grown_20260927154033403_abf")

In [ ]:
quick_patch_outputs = render_remaining_quick_patches()